# Integración del agente

**Fase:** E13

## Objetivo

Componer el prompt con diagnóstico, confianza, evidencia observacional, atribución y fragmentos recuperados, y obtener salida estructurada.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II reales, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

El LLM redacta y organiza; no diagnostica ni aporta conocimiento propio como fuente.

## Hipótesis

Un modelo de lenguaje pequeño (3B), local, al que se le entrega la decisión ya tomada y la evidencia ya verificada, puede **redactarla respetando el veredicto y citando solo lo que existe**, y un ajuste fino **solo del formato de la tarea** (sin contenido del manual en los pesos) mejora la tasa de salidas válidas sin introducir citas inventadas.

## Datos

- Nueve casos anotados de `configs/goldset_mazda3_bk.yaml` sobre ventanas de **validación** de `synthetic_v2` (el test sigue cerrado).
- Ajuste: 62 ejemplos derivados de ejecuciones auténticas del pipeline (55 entrenamiento, 7 validación), equilibrados ABSTAIN/EXPLAIN, reparto por caso. Ver informe E13.

## Método

1. **Antes:** Qwen2.5-3B-Instruct base, motor `transformers`, MPS, bf16, decodificación determinista.
2. **Ajuste:** LoRA rango 16 sobre `q/k/v/o_proj` (0,24 % de parámetros), 4 épocas, lr 2e-4, longitud 3 072, pérdida solo sobre la completion, semilla 20260901.
3. **Después:** el mismo motor cargando el adaptador. Mismas nueve preguntas, mismo prompt, mismo verificador.

El «antes» **no** es la medición con Ollama: aquella usaba un modelo cuantizado y otro motor, y no sería comparable.

## Métricas

Por caso, calculadas fuera del modelo por `run_agent` y `verify_citations`:

- **Salida válida:** JSON que cumple el esquema y no cita nada inexistente.
- **Respeta el veredicto:** redacta EXPLAIN si el verificador dijo EXPLAIN, ABSTAIN si dijo ABSTAIN.
- **Citas inventadas:** identificadores con forma correcta que no corresponden a nada entregado.
- **Esquema roto:** la salida no valida (y por qué).

## Criterio de éxito o refutación

Fijado antes de la segunda pasada (la primera reveló un fallo de esquema propio, corregido en `e2d7c59`; sus registros se conservan como `esquema_v1`):

- **Éxito:** después ≥ antes en salidas válidas, **cero** citas inventadas en ambos, y el ajustado no respeta el veredicto menos veces que el base.
- **Refutación:** el ajustado introduce citas inventadas, o respeta el veredicto menos que el base. Una mejora en validez que llegue a costa de cualquiera de las dos no cuenta.


In [ ]:
# Los notebooks importan del paquete; no duplican lógica científica.
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import mecanico_ia_exp

mecanico_ia_exp.__version__

## Ejecución

Las dos mediciones se lanzan desde la CLI y dejan un registro JSON por proveedor. El cuaderno **no** vuelve a generar: lee los registros y los resume con el paquete.

```bash
uv run mecanico-exp finetune-slm --epochs 4
uv run mecanico-exp sufficiency --llm transformers
uv run mecanico-exp sufficiency --llm transformers-lora
```

In [ ]:
import json

import pandas as pd

from mecanico_ia_exp.config import paths
from mecanico_ia_exp.llm.report import compare_records, per_case

REGISTROS = {
    "base (esquema v1)": paths.EXPERIMENTS / "E13_agente_transformers_esquema_v1.json",
    "base + LoRA (esquema v1)": paths.EXPERIMENTS / "E13_agente_transformers-lora_esquema_v1.json",
    "base": paths.EXPERIMENTS / "E13_agente_transformers.json",
    "base + LoRA": paths.EXPERIMENTS / "E13_agente_transformers-lora.json",
}
compare_records(REGISTROS)

In [ ]:
# Qué aprendió el adaptador, y cuánto: registro escrito por finetune().
historial = json.loads((paths.MODELS / "slm_lora" / "finetune_config.json").read_text())
h = historial["history"]
print(
    f"entrenables: {h['n_trainable_params'] / h['n_total_params']:.2%} · "
    f"{h['seconds'] / 60:.0f} min · recortados {h['n_truncated']}/{h['n_train']}"
)
pd.DataFrame(
    {
        "época": range(1, len(h["train_loss"]) + 1),
        "entrenamiento": h["train_loss"],
        "validación": h["val_loss"],
    }
).round(3)

In [ ]:
# Caso a caso, segunda pasada: dónde falla cada uno.
tabla = per_case({k: v for k, v in REGISTROS.items() if "v1" not in k})
tabla.pivot(index="caso", columns="proveedor", values=["redactó", "esquema"])

In [ ]:
# Texto literal de los fallos de esquema del ajustado, si los hay.
# El registro lo conserva para no tener que volver a generar.
ruta = REGISTROS["base + LoRA"]
if ruta.is_file():
    for r in json.loads(ruta.read_text())["runs"]:
        if r["schema_error"]:
            print(
                r["case_id"],
                "→",
                r["schema_error"].splitlines()[-2:],
                "\n  ",
                (r.get("raw_text") or "")[:200],
            )

## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
